In [1]:
import pandas as pd

#### Data Transformation

In [11]:
#  feature engineering

df = pd.read_csv("raw_data.csv")
df2 = df.copy()

# apply()
df2["tax"] = df2["income"].apply(lambda x : '20%' if x >= 50000 else '10%')

# map()
gender_map = {"Male":'M' , "Female":"F","Unknown":"U"}
df2["gender"] = df2["gender"].map(gender_map)

# assign()
df2 = df2.assign(new_income = df2["income"] * 1.1)

# replace(old,new)
df2["country"] = df2["country"].replace("USA","US")
df2

,id,name,age,country,gender,income,tax,new_income
0,1,John Doe,29.0,US,M,55000.0,20%,60500.0
1,1,John Doe,29.0,US,M,55000.0,20%,60500.0
2,2,Jane Smith,NaN,Canada,F,62000.0,20%,68200.0
3,3,Alex,NaN,US,U,47000.0,10%,51700.0
4,4,Maria Garcia,34.0,Spain,F,NaN,10%,NaN
5,5,Li Wei,27.0,China,M,51000.0,20%,56100.0
6,6,NaN,45.0,India,F,73000.0,20%,80300.0
7,7,Ahmed Khan,38.0,NaN,M,68000.0,20%,74800.0
8,8,Rachel Lee,29.0,US,F,62000.0,20%,68200.0
9,9,Carlos Ruiz,NaN,Mexico,M,45000.0,10%,49500.0


In [40]:
df

,id,name,age,country,gender,income
0,1,John Doe,29.0,USA,Male,55000.0
1,1,John Doe,29.0,USA,Male,55000.0
2,2,Jane Smith,NaN,Canada,Female,62000.0
3,3,Alex,NaN,USA,Unknown,47000.0
4,4,Maria Garcia,34.0,Spain,Female,NaN
5,5,Li Wei,27.0,China,Male,51000.0
6,6,NaN,45.0,India,Female,73000.0
7,7,Ahmed Khan,38.0,NaN,Male,68000.0
8,8,Rachel Lee,29.0,USA,Female,62000.0
9,9,Carlos Ruiz,NaN,Mexico,Male,45000.0


In [51]:
# Task - shifting 'id' column in the end

df2 = df.copy()
df2.columns = df2.columns.get_level_values(0) # for changing from multiIndex(tuple) to index (list)
# df2
# df2[["name","age","country","gender","income","id"]]  
new_col_order = [col for col in df2.columns if col != "id"] + ["id"]

print(new_col_order)
df2[new_col_order]



['name', 'age', 'country', 'gender', 'income', 'id']


,name,age,country,gender,income,id
0,John Doe,29.0,USA,Male,55000.0,1
1,John Doe,29.0,USA,Male,55000.0,1
2,Jane Smith,NaN,Canada,Female,62000.0,2
3,Alex,NaN,USA,Unknown,47000.0,3
4,Maria Garcia,34.0,Spain,Female,NaN,4
5,Li Wei,27.0,China,Male,51000.0,5
6,NaN,45.0,India,Female,73000.0,6
7,Ahmed Khan,38.0,NaN,Male,68000.0,7
8,Rachel Lee,29.0,USA,Female,62000.0,8
9,Carlos Ruiz,NaN,Mexico,Male,45000.0,9


In [57]:
# writing in csv file

df2 = df.copy()
df2.columns = df2.columns.get_level_values(0) # multiindex issue sol
df.columns = df.columns.get_level_values(0)

df2 = df2.drop_duplicates()
df2 = df2.fillna(0)
df2 = df2.sort_values("income")
df2 = df2.reset_index(drop = True)

df2.to_csv("sorted_data.csv")

In [68]:
# Grouping and aggregation of data

# groupby()
df.groupby("country")["income"].mean()
df.groupby("country")["income"].min()
df.groupby("country")["income"].max()


df.groupby("gender")["income"].mean()
df.groupby("gender")["income"].min()
df.groupby("gender")["income"].max()

# agg()
df.groupby("country")["income"].agg(["mean","min","max"])
df.groupby("country")["income"].aggregate(["mean","min","max"])  # does same as agg()


df.groupby("country")["income"].aggregate(avg_salary="mean",min_salary="min",max_salary="max")

# different column different aggregation
df.groupby("country").agg({
    "income" : "mean",
    "age" : "min"
})
# also written by renaming column
df.groupby("country").agg(
    avg_income=("income","mean"),
    min_age=("age","min")
)

,avg_income,min_age
country,,
Canada,62000.0,NaN
China,51000.0,27.0
India,73000.0,45.0
Mexico,45000.0,NaN
Spain,NaN,34.0
USA,55400.0,29.0


In [73]:
# Melt & Pivot

data = pd.DataFrame({
    "country" : ["USA","USA","India","India"],
    "year" : [2020,2021,2020,2021],
    "sales" : [100,120,90,110],
    "profit" : [20,25,18,22]
})

# melt - wide to long format
melted_data = data.melt(
    id_vars=["country","year"],
    value_vars=["sales","profit"],
    var_name="metrices",
    value_name="value"
)
melted_data

# pivot - long to wide format

original = melted_data.pivot(
    index=["country","year"],
    columns=["metrices"],
    values="value"
)
original

metrices      profit  sales
country year               
India   2020      18     90
        2021      22    110
USA     2020      20    100
        2021      25    120